# exp105_compact_rank_slot_features_on_exp098 train

Compact target-free PF/Beam rank-slot features on the exp098/exp073 full replay LightGBM surface.

## Contents

1. Setup and configuration
2. Input and feature plan
3. Rank-slot feature ablation training
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from compact_rank_slot_features_on_exp098 import run_compact_rank_slot_features_on_exp098
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_ROWS_ENV = os.environ.get("EXPERIMENT_MAX_ROWS")
MAX_TRAIN_ROWS_ENV = os.environ.get("EXPERIMENT_MAX_TRAIN_ROWS")

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

training_cfg = get_nested(config, "model.training") or {}
max_rows = int(MAX_ROWS_ENV) if MAX_ROWS_ENV else training_cfg.get("max_rows")
max_train_rows = (
    int(MAX_TRAIN_ROWS_ENV) if MAX_TRAIN_ROWS_ENV else training_cfg.get("max_train_rows")
)
if DEBUG:
    max_rows = max_rows or 20000
    max_train_rows = max_train_rows or 12000

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Status:", get_nested(config, "experiment.status"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Cache parent:", get_nested(config, "lineage.cache_parent"))
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "max_rows:", max_rows, "max_train_rows:", max_train_rows)

## 2. Input and feature plan

In [ ]:
rank_slot_cfg = get_nested(config, "model.rank_slot") or {}
variants = get_nested(config, "model.feature_ablation.active_variants") or []
modes = get_nested(config, "model.training.modes") or {}
active_modes = get_nested(config, "model.training.active_modes") or list(modes)

candidate_plan = pd.DataFrame(rank_slot_cfg.get("candidates") or [])
variant_plan = pd.DataFrame(
    [
        {
            "name": item.get("name"),
            "enabled": item.get("enabled", True),
            "feature_groups": ",".join(item.get("feature_groups") or []),
        }
        for item in variants
    ]
)
print("Rank-slot top_k:", rank_slot_cfg.get("top_k"))
display(candidate_plan)
display(variant_plan)
print("Active modes:", active_modes)
print("Kaggle sources:", get_nested(config, "runtime.kaggle.kernel_sources"))

## 3. Rank-slot feature ablation training

In [ ]:
summary = run_compact_rank_slot_features_on_exp098(
    output_dir=paths.artifacts_dir,
    train_dir=paths.train_data_dir,
    cache_path=get_nested(config, "data.exp072_train_feature_cache_local"),
    rank_slot_config=rank_slot_cfg,
    variants=variants,
    modes=modes,
    active_modes=active_modes,
    n_splits=int(get_nested(config, "validation.n_folds") or 5),
    fast=bool(get_nested(config, "audit.fast")),
    early_stopping_rounds=int(training_cfg.get("early_stopping_rounds", 250)),
    max_rows=max_rows,
    max_train_rows=max_train_rows,
    save_models=bool(training_cfg.get("save_models", True)),
    save_predictions=bool(training_cfg.get("save_predictions", True)),
    top_n_importance=int(training_cfg.get("top_n_importance", 50)),
)
print(json.dumps(summary, indent=2)[:4000])

## 4. Metrics and artifacts

In [ ]:
metrics_path = paths.artifacts_dir / "exp105_compact_rank_slot_features_on_exp098_metrics.csv"
summary_path = paths.artifacts_dir / "exp105_compact_rank_slot_features_on_exp098_summary.json"
rank_summary_path = paths.artifacts_dir / "exp105_compact_rank_slot_features_on_exp098_rank_slot_feature_summary.csv"

metrics = pd.read_csv(metrics_path)
pooled = metrics[metrics["fold"].astype(str).eq("pooled")].sort_values("rmse_tvt")
rank_summary = pd.read_csv(rank_summary_path)

paths.metrics_path.write_text(json.dumps(summary, indent=2) + "\n")
print("Summary:", summary_path)
print("Metrics mirror:", paths.metrics_path)
display(pooled[["variant", "mode", "model", "features", "rmse_tvt", "rmse_target"]].head(20))
display(rank_summary)